# 12 | LangGraph 多轮对话：checkpoint 和 thread_id

`thread_id` 标识一段会话。相同 `thread_id` 会恢复上一轮保存的消息历史；换一个 `thread_id` 就是新会话。

## 一、场景

旅行灵感记录器：用户分两轮补充偏好。

```text
第一轮：想找一个四天左右、适合放空的海边城市
第二轮：预算别太高，希望能看展
```

## 二、准备模型和图

本地模型使用 `qwen2.5:14b`。

`MessagesState` 会追加消息历史；节点只返回本轮新增的 AI 消息。

In [ ]:
from importlib.metadata import version
from typing import cast

from langchain_core.messages import AnyMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, END, MessagesState, StateGraph

print("langgraph", version("langgraph"))

In [ ]:
llm = ChatOllama(model="qwen2.5:14b", temperature=0)

system_message = SystemMessage(
    content=(
        "你是一个旅行灵感记录器。"
        "根据用户已经说过的偏好，整理一份很短的旅行灵感。"
        "回答控制在 3 条以内，不要编造具体价格。"
    )
)


def travel_recorder(state: MessagesState) -> dict[str, list[BaseMessage]]:
    history = cast(list[BaseMessage], state["messages"])
    messages: list[BaseMessage] = [system_message, *history]
    response = llm.invoke(messages)
    return {"messages": [response]}


builder = StateGraph(MessagesState)
builder.add_node("travel_recorder", travel_recorder)
builder.add_edge(START, "travel_recorder")
builder.add_edge("travel_recorder", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

## 三、第一轮：建立会话

第一次调用传入 `thread_id`，LangGraph 会保存本轮输入和模型回复。

In [ ]:
def show_last_ai_message(messages: list[AnyMessage]) -> None:
    ai_messages = [message for message in messages if message.type == "ai"]
    print(ai_messages[-1].content)


config: RunnableConfig = {"configurable": {"thread_id": "travel-ideas-001"}}

first_messages: list[AnyMessage] = [
    HumanMessage(content="我想找一个四天左右、适合放空的海边城市。")
]
first_input: MessagesState = {"messages": first_messages}
first_result = graph.invoke(first_input, config=config)
show_last_ai_message(cast(list[AnyMessage], first_result["messages"]))

## 四、第二轮：接上历史

第二次仍然使用同一个 `thread_id`，只需要传新消息；旧消息历史会从 checkpoint 恢复。

In [ ]:
second_messages: list[AnyMessage] = [
    HumanMessage(content="预算别太高，最好还能安排半天看展。")
]
second_input: MessagesState = {"messages": second_messages}

second_result = graph.invoke(second_input, config=config)
show_last_ai_message(cast(list[AnyMessage], second_result["messages"]))

## 五、换一个 `thread_id`

新的 `thread_id` 不会继承上一段会话的历史。

In [ ]:
other_config: RunnableConfig = {"configurable": {"thread_id": "travel-ideas-002"}}

other_messages: list[AnyMessage] = [
    HumanMessage(content="我也想安排一个轻松的小旅行。")
]
other_input: MessagesState = {"messages": other_messages}

other_result = graph.invoke(other_input, config=other_config)
show_last_ai_message(cast(list[AnyMessage], other_result["messages"]))

## 六、要点

- `checkpointer` 负责保存 State；
- `thread_id` 决定使用哪段会话历史；
- `MessagesState` 负责把新消息追加到旧消息后面。